In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

from io import BytesIO
from scipy.optimize import curve_fit
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.pipeline import make_pipeline

from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image as RLImage, PageBreak, HRFlowable
)

from pybaseball import batting_stats, statcast_sprint_speed

Importing and Adjusting Data

In [66]:
# =============================================================================
# DATA IMPORT
# =============================================================================

all_years = []
for year in range(2019, 2025):
    df = batting_stats(year, qual=100)
    df['Season'] = year
    all_years.append(df)

statcast = pd.concat(all_years, ignore_index=True)

sprint_all = []
for year in range(2019, 2025):
    df = statcast_sprint_speed(year)
    df['year'] = year
    sprint_all.append(df)

sprint = pd.concat(sprint_all, ignore_index=True)

sprint['last_name, first_name'] = sprint['last_name, first_name'].apply(
    lambda x: ' '.join(reversed(x.split(', '))) if isinstance(x, str) and ', ' in x else x
)
statcast = pd.merge(sprint, statcast, left_on=['last_name, first_name', 'year'],
                    right_on=['Name', 'Season'], how='left')

statcast['TB'] = ((statcast['H'] - statcast['2B'] - statcast['3B'] - statcast['HR'])
                  + statcast['2B'] * 2 + statcast['3B'] * 3 + statcast['HR'] * 4)
statcast['MB_Value'] = (((statcast['H'] + statcast['BB'] - statcast['CS']) * (statcast['TB'] + 0.7 * statcast['SB']))
                        / (statcast['AB'] + statcast['BB'] + statcast['CS']))

statcast.to_csv('Savant.csv')
sprint.to_csv('Sprint.csv')
# os.startfile('Savant.csv')
# os.startfile('Sprint.csv')

In [91]:
# =============================================================================
# CONFIG
# =============================================================================

BASE_FEATURES = [
    'O-Swing%', 'Z-Swing%', 'Swing%', 'O-Contact%', 'Z-Contact%', 'Contact%',
    'Zone%', 'F-Strike%', 'SwStr%', 'Barrel%', 'maxEV', 'HardHit', 'HardHit%',
    'CStr%', 'CSW%', 'EV', 'LA', 'bolts', 'hp_to_1b', 'sprint_speed'
]

REGRESSION_TARGETS = ['TB', 'BB', 'AB', 'H']
SB_CS_TARGETS      = ['SB', 'CS']
ALL_TARGETS        = REGRESSION_TARGETS + SB_CS_TARGETS
R2_THRESHOLD       = 0.8

PALETTE = {
    'TB': '#2980B9', 'BB': '#27AE60', 'AB': '#8E44AD', 'H': '#E67E22',
    'SB': '#E74C3C', 'CS': '#1ABC9C'
}
MODEL_COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628", "#1a9850"]

Research Data

In [92]:
# =============================================================================
# UTILITIES
# =============================================================================

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def adj_r2(r2, n, k):
    if n - k - 1 <= 0:
        return np.nan
    return round(1 - (1 - r2) * (n - 1) / (n - k - 1), 4)

def top_corr_features(data, target, features, top_n=5, min_abs=0.05):
    avail = [f for f in features if f in data.columns]
    corr = data[avail + [target]].corr()[target].drop(target).dropna()
    corr = corr[corr.abs() >= min_abs].abs().sort_values(ascending=False)
    return corr.index[:top_n].tolist()

def rl_image(buf, width_in, fig_w=7, fig_h=4.5):
    buf.seek(0)
    img = RLImage(buf, width=width_in * inch, height=width_in * inch * (fig_h / fig_w))
    img.hAlign = "CENTER"
    return img

def divider():
    return HRFlowable(width="100%", thickness=1, color=colors.HexColor("#cccccc"),
                      spaceAfter=8, spaceBefore=4)


Regression Models

In [93]:
# =============================================================================
# REGRESSION MODELS  (TB, BB, AB, H)
# =============================================================================

def run_regression_analysis(target, features, data):
    print(f"\n{'='*60}\n  TARGET: {target}\n{'='*60}")

    cols = features + [target]
    corr = data[cols].corr()[target]
    selected = corr[(corr.abs() > 0.3) & (corr.index != target)].index.tolist()
    print("Selected features:", selected)

    result = {'target': target, 'selected_features': selected, 'models': {}}
    if not selected:
        print(f"  No features with |r| > 0.3 for {target}. Skipping.")
        return result

    X = data[selected].dropna()
    y = data.loc[X.index, target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(X_train)
    Xte_s = scaler.transform(X_test)
    X_s   = scaler.fit_transform(X)

    # Linear
    lr = LinearRegression().fit(Xtr_s, y_train)
    lp = lr.predict(Xte_s)
    lc = cross_val_score(lr, X_s, y, cv=5, scoring='r2')
    result['models']['Linear Regression'] = {
        'r2': r2_score(y_test, lp), 'rmse': rmse(y_test, lp),
        'cv_r2_mean': lc.mean(), 'cv_r2_std': lc.std()
    }

    # Ridge
    rr = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5).fit(Xtr_s, y_train)
    rp = rr.predict(Xte_s)
    rc = cross_val_score(rr, X_s, y, cv=5, scoring='r2')
    result['models']['Ridge Regression'] = {
        'r2': r2_score(y_test, rp), 'rmse': rmse(y_test, rp),
        'cv_r2_mean': rc.mean(), 'cv_r2_std': rc.std(), 'best_alpha': rr.alpha_
    }

    # Lasso
    la = LassoCV(cv=5, random_state=42).fit(Xtr_s, y_train)
    lap = la.predict(Xte_s)
    lac = cross_val_score(la, X_s, y, cv=5, scoring='r2')
    result['models']['Lasso Regression'] = {
        'r2': r2_score(y_test, lap), 'rmse': rmse(y_test, lap),
        'cv_r2_mean': lac.mean(), 'cv_r2_std': lac.std(),
        'best_alpha': la.alpha_, 'coefficients': dict(zip(selected, la.coef_))
    }

    # Random Forest
    rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
    rfp = rf.predict(X_test)
    rfc = cross_val_score(rf, X, y, cv=5, scoring='r2')
    result['models']['Random Forest'] = {
        'r2': r2_score(y_test, rfp), 'rmse': rmse(y_test, rfp),
        'cv_r2_mean': rfc.mean(), 'cv_r2_std': rfc.std(),
        'feature_importances': dict(zip(selected, rf.feature_importances_))
    }

    for name, m in result['models'].items():
        print(f"  {name:<25} R²={m['r2']:.4f}  CV R²={m['cv_r2_mean']:.4f}  RMSE={m['rmse']:.4f}")

    return result


In [94]:
# =============================================================================
# SB / CS MODELS
# =============================================================================

def fit_sb_cs_models(data, target):
    best_feats = top_corr_features(data, target, BASE_FEATURES)
    if not best_feats:
        print(f"  No correlated features for {target}.")
        return [], None, None, None, None, None

    primary = best_feats[0]
    print(f"  {target}: primary={primary}, features={best_feats}")

    clean = data[[primary, target]].dropna()
    x = clean[primary].values
    y = clean[target].values.astype(float)
    x_plot = np.linspace(x.min(), x.max(), 300)

    results, curves = [], {}

    # 1. Linear
    lm = LinearRegression().fit(x.reshape(-1, 1), y)
    yp = lm.predict(x.reshape(-1, 1))
    r2 = r2_score(y, yp)
    results.append({"Model": "Linear", "R2": round(r2, 4), "AdjR2": adj_r2(r2, len(y), 1),
                    "RMSE": round(rmse(y, yp), 4), "Params": 1,
                    "Notes": "Baseline", "y_pred": yp})
    curves["Linear"] = lm.predict(x_plot.reshape(-1, 1))

    # 2. Sqrt
    xs = np.sqrt(np.maximum(x, 0))
    lm_sq = LinearRegression().fit(xs.reshape(-1, 1), y)
    yp = lm_sq.predict(xs.reshape(-1, 1))
    r2 = r2_score(y, yp)
    results.append({"Model": "Sqrt Transform", "R2": round(r2, 4), "AdjR2": adj_r2(r2, len(y), 1),
                    "RMSE": round(rmse(y, yp), 4), "Params": 1,
                    "Notes": f"{target} ~ sqrt({primary})", "y_pred": yp})
    curves["Sqrt Transform"] = lm_sq.predict(np.sqrt(np.maximum(x_plot, 0)).reshape(-1, 1))

    # 3. Log
    xl = np.log(np.maximum(x, 0.01))
    lm_log = LinearRegression().fit(xl.reshape(-1, 1), y)
    yp = lm_log.predict(xl.reshape(-1, 1))
    r2 = r2_score(y, yp)
    results.append({"Model": "Log Transform", "R2": round(r2, 4), "AdjR2": adj_r2(r2, len(y), 1),
                    "RMSE": round(rmse(y, yp), 4), "Params": 1,
                    "Notes": f"{target} ~ log({primary})", "y_pred": yp})
    curves["Log Transform"] = lm_log.predict(np.log(np.maximum(x_plot, 0.01)).reshape(-1, 1))

    # 4. Polynomial deg-2
    poly2 = make_pipeline(PolynomialFeatures(2), LinearRegression())
    poly2.fit(x.reshape(-1, 1), y)
    yp = poly2.predict(x.reshape(-1, 1))
    r2 = r2_score(y, yp)
    results.append({"Model": "Polynomial deg-2", "R2": round(r2, 4), "AdjR2": adj_r2(r2, len(y), 2),
                    "RMSE": round(rmse(y, yp), 4), "Params": 2,
                    "Notes": "Quadratic", "y_pred": yp})
    curves["Polynomial deg-2"] = poly2.predict(x_plot.reshape(-1, 1))

    # 5. Power Law / Neg Exponential
    if target == "SB":
        try:
            def power_law(xv, a, b): return a * np.power(np.maximum(xv, 0.01), b)
            popt, _ = curve_fit(power_law, x, y, p0=[1, 0.5], maxfev=8000)
            yp = power_law(x, *popt)
            r2 = r2_score(y, yp)
            results.append({"Model": "Power Law (a·x^b)", "R2": round(r2, 4),
                            "AdjR2": adj_r2(r2, len(y), 2), "RMSE": round(rmse(y, yp), 4),
                            "Params": 2, "Notes": f"a={popt[0]:.3f}, b={popt[1]:.3f}", "y_pred": yp})
            curves["Power Law (a·x^b)"] = power_law(x_plot, *popt)
        except Exception:
            pass
    else:
        try:
            def neg_exp(xv, a, b): return a * np.exp(b * xv)
            popt, _ = curve_fit(neg_exp, x, y, p0=[10, -1], maxfev=8000)
            yp = neg_exp(x, *popt)
            r2 = r2_score(y, yp)
            results.append({"Model": "Neg. Exponential (a·e^bx)", "R2": round(r2, 4),
                            "AdjR2": adj_r2(r2, len(y), 2), "RMSE": round(rmse(y, yp), 4),
                            "Params": 2, "Notes": f"a={popt[0]:.3f}, b={popt[1]:.3f}", "y_pred": yp})
            curves["Neg. Exponential (a·e^bx)"] = neg_exp(x_plot, *popt)
        except Exception:
            pass

    # 6. GLM
    glm_label = "Poisson GLM" if target == "SB" else "Neg. Binomial GLM"
    try:
        clean_m = data[best_feats + [target]].dropna()
        y_m   = clean_m[target].values.astype(float)
        X_raw = clean_m[best_feats].values.astype(float)
        scaler_m = StandardScaler()
        X_m  = np.column_stack([np.ones(len(X_raw)), scaler_m.fit_transform(X_raw)])
        beta = np.zeros(X_m.shape[1])
        for _ in range(100):
            mu = np.clip(np.exp(X_m @ beta), 1e-10, None)
            W  = np.diag(mu)
            z  = X_m @ beta + (y_m - mu) / mu
            try:
                beta = np.linalg.solve(X_m.T @ W @ X_m, X_m.T @ W @ z)
            except np.linalg.LinAlgError:
                break
        yp_m = np.exp(X_m @ beta)
        r2_m = r2_score(y_m, yp_m)
        results.append({
            "Model": f"{glm_label} ({len(best_feats)} features)",
            "R2": round(r2_m, 4), "AdjR2": adj_r2(r2_m, len(y_m), len(best_feats)),
            "RMSE": round(rmse(y_m, yp_m), 4), "Params": len(best_feats),
            "Notes": "Count GLM — log-link, multi-feature",
            "coefficients": dict(zip(["const"] + best_feats, beta))
        })
    except Exception as e:
        results.append({"Model": glm_label, "R2": np.nan, "AdjR2": np.nan,
                        "RMSE": np.nan, "Params": 0, "Notes": f"Failed: {e}"})

    best = max((r for r in results if not np.isnan(r.get("R2", np.nan))), key=lambda r: r["R2"])
    print(f"  {target} best: {best['Model']} (R²={best['R2']})")
    return results, primary, x, y, x_plot, curves


def make_scatter_plot(x, y, results, x_plot, curves, primary, target):
    color = PALETTE[target]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(x, y, alpha=0.25, s=15, color=color, label="Observations")
    valid_r2 = {r["Model"]: r["R2"] for r in results if not np.isnan(r.get("R2", np.nan))}
    for i, (name, curve) in enumerate(curves.items()):
        r2 = valid_r2.get(name)
        lbl = f"{name} (R²={r2})" if r2 is not None else name
        ax.plot(x_plot, curve, color=MODEL_COLORS[i % len(MODEL_COLORS)], lw=2, label=lbl)
    ax.set_xlabel(primary, fontsize=10)
    ax.set_ylabel(target, fontsize=10)
    ax.set_title(f"{target} vs {primary} — Model Fits", fontsize=12, fontweight="bold", color=color)
    ax.legend(fontsize=7.5, loc="best")
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150)
    plt.close(fig)
    return buf


def make_residual_plot(x, y, results, primary, target):
    valid = [r for r in results if not np.isnan(r.get("R2", np.nan)) and "y_pred" in r]
    if not valid:
        return None
    cols = 3
    rows = (len(valid) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(9, rows * 2.8))
    axes = axes.flatten()
    for i, r in enumerate(valid):
        ax = axes[i]
        ax.scatter(x, r["y_pred"] - y, alpha=0.25, s=10, color=MODEL_COLORS[i % len(MODEL_COLORS)])
        ax.axhline(0, color="black", lw=1.2, linestyle="--")
        ax.set_title(r["Model"], fontsize=8, fontweight="bold")
        ax.set_xlabel(primary, fontsize=7)
        ax.set_ylabel("Residual", fontsize=7)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    fig.suptitle(f"{target} — Residual Plots", fontsize=10, fontweight="bold")
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf


In [95]:
# =============================================================================
# DEPLOYMENT
# =============================================================================

def get_best_regression_model(reg_result, threshold=R2_THRESHOLD):
    if not reg_result.get("models"):
        return None, None
    best_name, best_m = max(reg_result["models"].items(), key=lambda x: x[1]["r2"])
    return (best_name, best_m) if best_m["r2"] >= threshold else (None, None)


def get_best_sb_cs_model(sb_cs_result, threshold=R2_THRESHOLD):
    valid = [r for r in sb_cs_result.get("results", []) if not np.isnan(r.get("R2", np.nan))]
    if not valid:
        return None
    best = max(valid, key=lambda r: r["R2"])
    return best if best["R2"] >= threshold else None


def _refit_and_predict(model_name, reg_result, df_deploy):
    feats  = reg_result["selected_features"]
    target = reg_result["target"]
    avail  = [f for f in feats if f in df_deploy.columns and f in statcast.columns]
    if not avail:
        return None

    train_clean = statcast[avail + [target]].dropna()
    X_train  = train_clean[avail].values
    y_train  = train_clean[target].values
    has_data = df_deploy[avail].notna().all(axis=1)
    X_deploy = df_deploy[avail].values

    scaler = StandardScaler()
    Xtr_s  = scaler.fit_transform(X_train)
    Xde_s  = scaler.transform(X_deploy)

    try:
        if model_name == "Linear Regression":
            m = LinearRegression().fit(Xtr_s, y_train)
        elif model_name == "Ridge Regression":
            m = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5).fit(Xtr_s, y_train)
        elif model_name == "Lasso Regression":
            m = LassoCV(cv=5, random_state=42).fit(Xtr_s, y_train)
        elif model_name == "Random Forest":
            m = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
            preds = np.full(len(X_deploy), np.nan)
            preds[has_data] = m.predict(X_deploy[has_data])
            return preds
        else:
            return None

        preds = np.full(len(Xde_s), np.nan)
        preds[has_data] = m.predict(Xde_s[has_data])
        return preds

    except Exception as e:
        print(f"    WARNING: Refit failed for {model_name} ({e})")
        return None


def deploy_all(df_deploy, all_results_reg, all_results_sb_cs):
    result_df = df_deploy.copy()

    # TB, BB, AB, H
    for res in all_results_reg:
        target   = res["target"]
        col_name = f"x{target}"
        model_name, model_dict = get_best_regression_model(res)

        if model_name is None:
            best_r2 = max((m["r2"] for m in res["models"].values()), default=None)
            print(f"  [{target}] Best R²={best_r2:.4f} < {R2_THRESHOLD}. x{target} = actual.")
            result_df[col_name] = result_df.get(target, np.nan)
            continue

        print(f"  [{target}] Deploying '{model_name}' (R²={model_dict['r2']:.4f}) → '{col_name}'")
        preds = _refit_and_predict(model_name, res, result_df)

        if preds is None:
            result_df[col_name] = result_df.get(target, np.nan)
        else:
            result_df[col_name] = np.where(np.isnan(preds), result_df.get(target, np.nan), preds)
            result_df[col_name] = result_df[col_name].clip(lower=0)

    # SB, CS
    for target in SB_CS_TARGETS:
        col_name = f"x{target}"
        data     = all_results_sb_cs.get(target, {})
        primary  = data.get("primary")
        best     = get_best_sb_cs_model(data)

        if best is None:
            valid   = [r for r in data.get("results", []) if not np.isnan(r.get("R2", np.nan))]
            best_r2 = max((r["R2"] for r in valid), default=None)
            print(f"  [{target}] Best R²={best_r2} < {R2_THRESHOLD}. x{target} = actual.")
            result_df[col_name] = result_df.get(target, np.nan)
            continue

        model_name = best["Model"]
        print(f"  [{target}] Deploying '{model_name}' (R²={best['R2']}) → '{col_name}'")

        if "GLM" in model_name and "coefficients" in best:
            coef_dict     = best["coefficients"]
            features_used = [k for k in coef_dict if k != "const"]
            avail         = [f for f in features_used if f in result_df.columns]
            if not avail:
                result_df[col_name] = result_df.get(target, np.nan)
                continue
            X_raw    = result_df[avail].values.astype(float)
            X_scaled = StandardScaler().fit_transform(X_raw)
            beta     = np.array([coef_dict.get("const", 0)] + [coef_dict[f] for f in avail])
            X_design = np.hstack([np.ones((len(X_scaled), 1)), X_scaled])
            result_df[col_name] = np.exp(X_design @ beta).clip(0)

        elif primary is None or primary not in result_df.columns:
            result_df[col_name] = result_df.get(target, np.nan)

        else:
            x_vals  = result_df[primary].values.astype(float)
            x_train = statcast[primary].dropna().values
            y_train = statcast.loc[statcast[primary].notna(), target].values.astype(float)
            try:
                if model_name == "Linear":
                    m_c, b_c = np.polyfit(x_train, y_train, 1)
                    result_df[col_name] = (m_c * x_vals + b_c).clip(0)
                elif model_name == "Sqrt Transform":
                    lm = LinearRegression().fit(np.sqrt(np.maximum(x_train, 0)).reshape(-1, 1), y_train)
                    result_df[col_name] = lm.predict(np.sqrt(np.maximum(x_vals, 0)).reshape(-1, 1)).clip(0)
                elif model_name == "Log Transform":
                    lm = LinearRegression().fit(np.log(np.maximum(x_train, 0.01)).reshape(-1, 1), y_train)
                    result_df[col_name] = lm.predict(np.log(np.maximum(x_vals, 0.01)).reshape(-1, 1)).clip(0)
                elif model_name == "Polynomial deg-2":
                    poly = make_pipeline(PolynomialFeatures(2), LinearRegression())
                    poly.fit(x_train.reshape(-1, 1), y_train)
                    result_df[col_name] = poly.predict(x_vals.reshape(-1, 1)).clip(0)
                elif "Power Law" in model_name:
                    def power_law(xv, a, b): return a * np.power(np.maximum(xv, 0.01), b)
                    popt, _ = curve_fit(power_law, x_train, y_train, p0=[1, 0.5], maxfev=8000)
                    result_df[col_name] = power_law(x_vals, *popt).clip(0)
                elif "Exponential" in model_name:
                    def neg_exp(xv, a, b): return a * np.exp(b * xv)
                    popt, _ = curve_fit(neg_exp, x_train, y_train, p0=[10, -1], maxfev=8000)
                    result_df[col_name] = neg_exp(x_vals, *popt).clip(0)
                else:
                    result_df[col_name] = result_df.get(target, np.nan)
            except Exception as e:
                print(f"    WARNING: {e}. Falling back to actual.")
                result_df[col_name] = result_df.get(target, np.nan)

    return result_df


In [96]:
# =============================================================================
# COMBINED PDF REPORT
# =============================================================================

def build_combined_pdf(all_results_reg, all_results_sb_cs, deployed_df, output_path):
    doc = SimpleDocTemplate(
        output_path, pagesize=letter,
        leftMargin=0.75*inch, rightMargin=0.75*inch,
        topMargin=0.75*inch, bottomMargin=0.75*inch
    )
    styles = getSampleStyleSheet()
    S = {
        "title":   ParagraphStyle("T",    parent=styles["Title"],    fontSize=20, alignment=TA_CENTER, spaceAfter=6),
        "h1":      ParagraphStyle("H1",   parent=styles["Heading1"], fontSize=14, textColor=colors.HexColor("#2C3E50"), spaceBefore=12, spaceAfter=6),
        "h2":      ParagraphStyle("H2",   parent=styles["Heading2"], fontSize=11, textColor=colors.HexColor("#2980B9"), spaceBefore=8,  spaceAfter=4),
        "body":    ParagraphStyle("Body", parent=styles["Normal"],   fontSize=9.5, leading=14, spaceAfter=5),
        "small":   ParagraphStyle("Sm",   parent=styles["Normal"],   fontSize=8.5, leading=12, textColor=colors.HexColor("#444444")),
        "caption": ParagraphStyle("Cap",  parent=styles["Normal"],   fontSize=8,   leading=11, textColor=colors.grey, alignment=TA_CENTER, spaceAfter=8),
        "header":  ParagraphStyle("TH",   parent=styles["Normal"],   textColor=colors.white, fontName="Helvetica-Bold", fontSize=9, alignment=TA_CENTER),
    }

    def th(text): return Paragraph(text, S["header"])

    def styled_table(rows, col_widths, header_color="#2C3E50", highlight_row=None):
        t = Table(rows, colWidths=col_widths, repeatRows=1)
        cmds = [
            ("BACKGROUND",    (0, 0), (-1, 0),  colors.HexColor(header_color)),
            ("TEXTCOLOR",     (0, 0), (-1, 0),  colors.white),
            ("FONTSIZE",      (0, 0), (-1, -1), 8),
            ("ALIGN",         (0, 0), (-1, -1), "CENTER"),
            ("ROWBACKGROUNDS",(0, 1), (-1, -1), [colors.HexColor("#ECF0F1"), colors.white]),
            ("GRID",          (0, 0), (-1, -1), 0.5, colors.grey),
            ("TOPPADDING",    (0, 0), (-1, -1), 4),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
        ]
        if highlight_row is not None:
            cmds += [("BACKGROUND", (0, highlight_row), (-1, highlight_row), colors.HexColor("#d4edda")),
                     ("FONTNAME",   (0, highlight_row), (-1, highlight_row), "Helvetica-Bold")]
        t.setStyle(TableStyle(cmds))
        return t

    story = []

    # ── Title page ─────────────────────────────────────────────────────────
    story.append(Spacer(1, 0.8*inch))
    story.append(Paragraph("Baseball Statcast", S["title"]))
    story.append(Paragraph("Full Model Report  (2019–2024)", S["title"]))
    story.append(Spacer(1, 0.2*inch))
    story.append(divider())
    story.append(Spacer(1, 0.15*inch))
    story.append(Paragraph(
        "This report covers all six target variables: <b>TB, BB, AB, H</b> (linear models) and "
        "<b>SB, CS</b> (count/nonlinear models). Each section shows model comparisons and feature details. "
        f"Part 3 shows expected values — models are only deployed if R² ≥ <b>{R2_THRESHOLD}</b>; "
        "otherwise the expected value equals the actual observed value.", S["body"]))
    story.append(PageBreak())

    # ── Overall summary ────────────────────────────────────────────────────
    story.append(Paragraph("Overall Best Model Summary", S["h1"]))
    story.append(divider())
    summary_rows = [[th("Target"), th("Best Model"), th("R²"), th("CV R²"), th("RMSE"), th("Deployed?")]]

    for res in all_results_reg:
        if not res["models"]:
            continue
        best_name, best_m = max(res["models"].items(), key=lambda x: x[1]["r2"])
        deployed = "Yes" if best_m["r2"] >= R2_THRESHOLD else "No (actual used)"
        summary_rows.append([res["target"], best_name, f"{best_m['r2']:.4f}",
                             f"{best_m['cv_r2_mean']:.4f} ± {best_m['cv_r2_std']:.4f}",
                             f"{best_m['rmse']:.4f}", deployed])

    for target, data in all_results_sb_cs.items():
        valid = [r for r in data["results"] if not np.isnan(r.get("R2", np.nan))]
        if not valid:
            continue
        best = max(valid, key=lambda r: r["R2"])
        deployed = "Yes" if best["R2"] >= R2_THRESHOLD else "No (actual used)"
        summary_rows.append([target, best["Model"], str(best["R2"]),
                             str(best.get("AdjR2", "N/A")), str(best["RMSE"]), deployed])

    story.append(styled_table(summary_rows,
        [0.6*inch, 2.0*inch, 0.65*inch, 1.7*inch, 0.75*inch, 1.1*inch]))
    story.append(PageBreak())

    # ── Part 1: Regression targets ─────────────────────────────────────────
    story.append(Paragraph("Part 1 — Regression Targets: TB, BB, AB, H", S["h1"]))
    story.append(divider())
    story.append(Paragraph(
        "Linear, Ridge, Lasso, and Random Forest models were evaluated. "
        "Features selected by |correlation| > 0.3 with each target.", S["body"]))

    for res in all_results_reg:
        target = res["target"]
        story.append(Paragraph(f"Target: {target}", S["h2"]))

        if not res["selected_features"]:
            story.append(Paragraph("No features met the correlation threshold.", S["body"]))
            continue

        story.append(Paragraph(f"Selected features: {', '.join(res['selected_features'])}", S["small"]))
        story.append(Spacer(1, 0.1*inch))

        model_rows = [[th("Model"), th("R²"), th("CV R²"), th("RMSE"), th("Alpha")]]
        best_r2 = max(m["r2"] for m in res["models"].values())
        highlight = None
        for i, (name, m) in enumerate(res["models"].items(), start=1):
            alpha = f"{m['best_alpha']:.4f}" if isinstance(m.get("best_alpha"), float) else "N/A"
            model_rows.append([name, f"{m['r2']:.4f}",
                               f"{m['cv_r2_mean']:.4f} ± {m['cv_r2_std']:.4f}",
                               f"{m['rmse']:.4f}", alpha])
            if m["r2"] == best_r2:
                highlight = i
        story.append(styled_table(model_rows,
            [1.7*inch, 0.75*inch, 1.8*inch, 0.75*inch, 0.85*inch],
            header_color="#2980B9", highlight_row=highlight))
        story.append(Spacer(1, 0.15*inch))

        lasso = res["models"].get("Lasso Regression", {})
        if "coefficients" in lasso:
            story.append(Paragraph("Lasso Coefficients", S["h2"]))
            coef_rows = [[th("Feature"), th("Coefficient")]]
            for feat, coef in sorted(lasso["coefficients"].items(), key=lambda x: abs(x[1]), reverse=True):
                coef_rows.append([feat, f"{coef:.4f}"])
            story.append(styled_table(coef_rows, [3.5*inch, 1.5*inch], header_color="#2980B9"))
            story.append(Spacer(1, 0.15*inch))

        rf = res["models"].get("Random Forest", {})
        if "feature_importances" in rf:
            story.append(Paragraph("Random Forest Feature Importances", S["h2"]))
            imp_rows = [[th("Feature"), th("Importance")]]
            for feat, imp in sorted(rf["feature_importances"].items(), key=lambda x: x[1], reverse=True):
                imp_rows.append([feat, f"{imp:.4f}"])
            story.append(styled_table(imp_rows, [3.5*inch, 1.5*inch], header_color="#2980B9"))

        story.append(PageBreak())

    # ── Part 2: SB / CS ────────────────────────────────────────────────────
    story.append(Paragraph("Part 2 — Count Targets: SB, CS", S["h1"]))
    story.append(divider())
    story.append(Paragraph(
        "SB and CS are non-negative integer counts with nonlinear relationships to speed metrics. "
        "Six model families were tested: Linear, Sqrt transform, Log transform, Polynomial deg-2, "
        "Power Law / Negative Exponential, and a count GLM (Poisson for SB; Neg. Binomial for CS).",
        S["body"]))

    for target, data in all_results_sb_cs.items():
        if not data["results"]:
            continue
        story.append(Paragraph(f"Target: {target}", S["h2"]))
        story.append(Paragraph(f"Primary predictor: <b>{data['primary']}</b>", S["small"]))
        story.append(Spacer(1, 0.1*inch))

        valid = [r for r in data["results"] if not np.isnan(r.get("R2", np.nan))]
        best_r2 = max((r["R2"] for r in valid), default=None)

        model_rows = [[th("Model"), th("R²"), th("Adj R²"), th("RMSE"), th("Notes")]]
        highlight = None
        for i, r in enumerate(sorted(valid, key=lambda x: -x["R2"]), start=1):
            model_rows.append([r["Model"], str(r["R2"]), str(r.get("AdjR2", "N/A")),
                               str(r["RMSE"]), r.get("Notes", "")])
            if r["R2"] == best_r2 and highlight is None:
                highlight = i
        story.append(styled_table(model_rows,
            [1.9*inch, 0.6*inch, 0.7*inch, 0.65*inch, 2.0*inch],
            highlight_row=highlight))
        story.append(Paragraph("Green row = best R².", S["caption"]))

        if data.get("scatter_buf"):
            story.append(Paragraph("Fitted Curves", S["h2"]))
            story.append(rl_image(data["scatter_buf"], 6.3))
        if data.get("resid_buf"):
            story.append(Paragraph("Residual Plots", S["h2"]))
            story.append(rl_image(data["resid_buf"], 6.3, fig_w=9, fig_h=5.6))

        for r in [r for r in data["results"] if "GLM" in r.get("Model", "") and "coefficients" in r]:
            story.append(Paragraph(f"{r['Model']} — Coefficients", S["h2"]))
            coef_rows = [[th("Feature"), th("Coefficient")]]
            for feat, val in sorted(r["coefficients"].items(), key=lambda kv: abs(kv[1]), reverse=True):
                coef_rows.append([feat, f"{val:.4f}"])
            story.append(styled_table(coef_rows, [3.5*inch, 1.5*inch], header_color="#2980B9"))

        story.append(PageBreak())

    # ── Part 3: Expected Values ────────────────────────────────────────────
    story.append(Paragraph("Part 3 — Expected Values (Model Deployment)", S["h1"]))
    story.append(divider())
    story.append(Paragraph(
        f"Best models with R² ≥ {R2_THRESHOLD} were deployed to calculate expected values. "
        "Where no model met the threshold, the expected value equals the actual observed value. "
        "Full results are saved to <b>expected_all_stats.csv</b>.", S["body"]))
    story.append(Spacer(1, 0.15*inch))

    disp_cols = [c for c in
        ["Name", "Season", "TB", "xTB", "BB", "xBB", "AB", "xAB", "H", "xH", "SB", "xSB", "CS", "xCS"]
        if c in deployed_df.columns]

    preview = deployed_df[disp_cols].dropna(subset=["Name"]).head(30).round(2)
    col_w = [1.3*inch, 0.45*inch] + [0.52*inch] * (len(disp_cols) - 2)
    dep_rows = [[th(c) for c in disp_cols]]
    for _, row in preview.iterrows():
        dep_rows.append([str(row[c]) for c in disp_cols])

    story.append(styled_table(dep_rows, col_w))
    story.append(Paragraph("Showing first 30 rows. Full results saved to expected_all_stats.csv.", S["caption"]))

    doc.build(story)
    print(f"\nCombined PDF saved to: {output_path}")


In [97]:
# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":

    # --- Regression targets (TB, BB, AB, H) ---
    all_results_reg = []
    for target in REGRESSION_TARGETS:
        result = run_regression_analysis(target, BASE_FEATURES, statcast)
        all_results_reg.append(result)

    # --- SB / CS targets ---
    all_results_sb_cs = {}
    for target in SB_CS_TARGETS:
        print(f"\nFitting models for {target}...")
        pack = fit_sb_cs_models(statcast, target)
        results, primary, x, y, x_plot, curves = pack

        if not results:
            all_results_sb_cs[target] = {"results": [], "primary": None}
            continue

        scatter_buf = make_scatter_plot(x, y, results, x_plot, curves, primary, target)
        resid_buf   = make_residual_plot(x, y, results, primary, target)

        all_results_sb_cs[target] = {
            "results": results, "primary": primary,
            "scatter_buf": scatter_buf, "resid_buf": resid_buf,
        }

    # --- Deploy models ---
    print("\n" + "="*60)
    print("  DEPLOYING MODELS")
    print("="*60)
    deployed_df = deploy_all(statcast, all_results_reg, all_results_sb_cs)

    # Save CSV
    preview_cols = [c for c in
        ["Name", "Season", "TB", "xTB", "BB", "xBB", "AB", "xAB", "H", "xH", "SB", "xSB", "CS", "xCS"]
        if c in deployed_df.columns]
    deployed_df[preview_cols].to_csv("expected_all_stats.csv", index=False)
    print("Saved: expected_all_stats.csv")

    # --- Build single combined PDF ---
    OUTPUT_PDF = "statcast_full_report.pdf"
    build_combined_pdf(all_results_reg, all_results_sb_cs, deployed_df, OUTPUT_PDF)
    os.startfile(OUTPUT_PDF)


  TARGET: TB
Selected features: ['Barrel%', 'maxEV', 'HardHit', 'HardHit%', 'CSW%', 'EV']
  Linear Regression         R²=0.9486  CV R²=0.9414  RMSE=18.0466
  Ridge Regression          R²=0.9486  CV R²=0.9412  RMSE=18.0462
  Lasso Regression          R²=0.9487  CV R²=0.9413  RMSE=18.0398
  Random Forest             R²=0.9469  CV R²=0.9407  RMSE=18.3472

  TARGET: BB
Selected features: ['O-Swing%', 'Swing%', 'F-Strike%', 'Barrel%', 'maxEV', 'HardHit', 'HardHit%', 'EV']
  Linear Regression         R²=0.8115  CV R²=0.7984  RMSE=9.0688
  Ridge Regression          R²=0.8115  CV R²=0.7983  RMSE=9.0687
  Lasso Regression          R²=0.8115  CV R²=0.7983  RMSE=9.0695
  Random Forest             R²=0.8491  CV R²=0.8416  RMSE=8.1139

  TARGET: AB
Selected features: ['maxEV', 'HardHit', 'CSW%']
  Linear Regression         R²=0.9044  CV R²=0.9012  RMSE=49.0419
  Ridge Regression          R²=0.9044  CV R²=0.9012  RMSE=49.0370
  Lasso Regression          R²=0.9045  CV R²=0.9012  RMSE=49.0237
  Rando

In [98]:
MB = pd.read_csv('expected_all_stats.csv')
MB['xMB'] = round(((MB['xH'] + MB['xBB'] - MB['xCS']) * (MB['xTB'] + 0.7 * MB['xSB'])) / (MB['xAB'] + MB['xBB'] + MB['xCS']))
MB['MB'] = round(((MB['H'] + MB['BB'] - MB['CS']) * (MB['TB'] + 0.7 * MB['SB'])) / (MB['AB'] + MB['BB'] + MB['CS']))
MB.to_csv('expected_all_stats.csv')
# os.startfile('expected_all_stats.csv')

# Comparison
Compare the Outcomes of the MB and xMB values (Find a 'mathematical way')

In [99]:
# =============================================================================
# HELPERS
# =============================================================================

def divider():
    return HRFlowable(width="100%", thickness=1,
                      color=colors.HexColor("#cccccc"),
                      spaceAfter=8, spaceBefore=4)

def rl_image(buf, width_in=6.5, fig_w=8, fig_h=4.5):
    buf.seek(0)
    img = RLImage(buf,
                  width=width_in * inch,
                  height=width_in * inch * (fig_h / fig_w))
    img.hAlign = "CENTER"
    return img

def th(text, S):
    return Paragraph(text, S["header"])

def styled_table(rows, col_widths, header_color="#2C3E50", highlight_row=None):
    t = Table(rows, colWidths=col_widths, repeatRows=1)
    cmds = [
        ("BACKGROUND",     (0, 0),  (-1, 0),  colors.HexColor(header_color)),
        ("TEXTCOLOR",      (0, 0),  (-1, 0),  colors.white),
        ("FONTSIZE",       (0, 0),  (-1, -1), 8.5),
        ("ALIGN",          (0, 0),  (-1, -1), "CENTER"),
        ("ROWBACKGROUNDS", (0, 1),  (-1, -1), [colors.HexColor("#ECF0F1"), colors.white]),
        ("GRID",           (0, 0),  (-1, -1), 0.5, colors.grey),
        ("TOPPADDING",     (0, 0),  (-1, -1), 4),
        ("BOTTOMPADDING",  (0, 0),  (-1, -1), 4),
        ("LEFTPADDING",    (0, 0),  (-1, -1), 6),
        ("RIGHTPADDING",   (0, 0),  (-1, -1), 6),
    ]
    if highlight_row is not None:
        cmds += [
            ("BACKGROUND", (0, highlight_row), (-1, highlight_row),
             colors.HexColor("#d4edda")),
            ("FONTNAME",   (0, highlight_row), (-1, highlight_row),
             "Helvetica-Bold"),
        ]
    t.setStyle(TableStyle(cmds))
    return t

In [100]:
# =============================================================================
# ANALYSIS
# =============================================================================

def run_analysis(csv_path="expected_all_stats.csv"):
    df = pd.read_csv(csv_path)

    # Recompute MB / xMB
    df["xMB"] = round(
        ((df["xH"] + df["xBB"] - df["xCS"]) * (df["xTB"] + 0.7 * df["xSB"]))
        / (df["xAB"] + df["xBB"] + df["xCS"])
    )
    df["MB"] = round(
        ((df["H"] + df["BB"] - df["CS"]) * (df["TB"] + 0.7 * df["SB"]))
        / (df["AB"] + df["BB"] + df["CS"])
    )

    clean = df[["Name", "Season", "MB", "xMB"]].dropna()
    clean = clean[clean["MB"] != 0].copy()
    n = len(clean)
    y    = clean["MB"].values.astype(float)
    xhat = clean["xMB"].values.astype(float)

    # ── Percent Error ─────────────────────────────────────────────────────
    clean["Percent_Error"]     = ((xhat - y) / np.abs(y)) * 100
    clean["Abs_Percent_Error"] = clean["Percent_Error"].abs()
    pe  = clean["Percent_Error"]
    ape = clean["Abs_Percent_Error"]

    pe_stats = {
        "n":                n,
        "mean_pe":          round(pe.mean(),    2),
        "median_pe":        round(pe.median(),  2),
        "mean_ape":         round(ape.mean(),   2),
        "median_ape":       round(ape.median(), 2),
        "std_pe":           round(pe.std(),     2),
        "min_pe":           round(pe.min(),     2),
        "max_pe":           round(pe.max(),     2),
        "pct_within_5":     round((ape <= 5).mean()  * 100, 1),
        "pct_within_10":    round((ape <= 10).mean() * 100, 1),
        "pct_within_20":    round((ape <= 20).mean() * 100, 1),
        "top_over":         clean.nlargest(5,  "Percent_Error"),
        "top_under":        clean.nsmallest(5, "Percent_Error"),
    }

    # ── Bayesian Model Comparison ─────────────────────────────────────────
    # Model 0: MB ~ mean(MB)
    y_pred_0 = np.full_like(y, y.mean())
    rss_0    = np.sum((y - y_pred_0) ** 2)
    k_0      = 1
    bic_0    = n * np.log(rss_0 / n) + k_0 * np.log(n)

    # Model 1: MB ~ a + b*xMB  (OLS)
    X_des   = np.column_stack([np.ones(n), xhat])
    beta, _, _, _ = np.linalg.lstsq(X_des, y, rcond=None)
    y_pred_1 = X_des @ beta
    rss_1    = np.sum((y - y_pred_1) ** 2)
    k_1      = 2
    bic_1    = n * np.log(rss_1 / n) + k_1 * np.log(n)

    delta_bic    = bic_0 - bic_1
    bayes_factor = np.exp(delta_bic / 2)
    posterior_m1 = bayes_factor / (1 + bayes_factor)
    ss_tot       = np.sum((y - y.mean()) ** 2)
    r2_m1        = 1 - rss_1 / ss_tot

    def jeffreys_label(bf):
        if bf < 1:   return "Against Model 1 (xMB adds no predictive value)"
        elif bf < 3:  return "Anecdotal evidence for Model 1"
        elif bf < 10: return "Moderate evidence for Model 1"
        elif bf < 30: return "Strong evidence for Model 1"
        elif bf < 100:return "Very Strong evidence for Model 1"
        else:         return "Decisive evidence for Model 1"

    bayes_stats = {
        "bic_0":         round(bic_0,         2),
        "bic_1":         round(bic_1,         2),
        "delta_bic":     round(delta_bic,     2),
        "bayes_factor":  round(bayes_factor,  2),
        "log10_bf":      round(np.log10(max(bayes_factor, 1e-10)), 2),
        "posterior_m1":  round(posterior_m1 * 100, 1),
        "intercept":     round(beta[0],       3),
        "slope":         round(beta[1],       3),
        "r2_m1":         round(r2_m1,         4),
        "label":         jeffreys_label(bayes_factor),
    }

    return clean, pe_stats, bayes_stats

# =============================================================================
# FIGURES
# =============================================================================

def fig_percent_error_dist(clean):
    """Histogram + box plot of percent error."""
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    pe = clean["Percent_Error"]

    # Histogram
    ax = axes[0]
    ax.hist(pe, bins=40, color="#2980B9", edgecolor="white", alpha=0.85)
    ax.axvline(0,          color="black",  lw=1.5, linestyle="--", label="Zero error")
    ax.axvline(pe.mean(),  color="#E74C3C", lw=1.5, linestyle="-",  label=f"Mean ({pe.mean():.1f}%)")
    ax.axvline(pe.median(),color="#27AE60", lw=1.5, linestyle="-",  label=f"Median ({pe.median():.1f}%)")
    ax.set_xlabel("Percent Error (%)", fontsize=10)
    ax.set_ylabel("Count", fontsize=10)
    ax.set_title("Distribution of Percent Error", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

    # Box plot by season
    ax2 = axes[1]
    seasons = sorted(clean["Season"].unique())
    data_by_season = [clean.loc[clean["Season"] == s, "Percent_Error"].values for s in seasons]
    bp = ax2.boxplot(data_by_season, patch_artist=True, notch=False,
                     medianprops=dict(color="black", lw=2))
    for patch in bp["boxes"]:
        patch.set_facecolor("#2980B9")
        patch.set_alpha(0.7)
    ax2.axhline(0, color="#E74C3C", lw=1.2, linestyle="--")
    ax2.set_xticklabels(seasons, fontsize=9)
    ax2.set_xlabel("Season", fontsize=10)
    ax2.set_ylabel("Percent Error (%)", fontsize=10)
    ax2.set_title("Percent Error by Season", fontsize=11, fontweight="bold")
    ax2.grid(True, alpha=0.3, axis="y")
    ax2.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf


def fig_mb_vs_xmb(clean):
    """Scatter: MB vs xMB with OLS line."""
    y    = clean["MB"].values.astype(float)
    xhat = clean["xMB"].values.astype(float)
    n    = len(clean)

    X_des = np.column_stack([np.ones(n), xhat])
    beta, _, _, _ = np.linalg.lstsq(X_des, y, rcond=None)
    x_line = np.linspace(xhat.min(), xhat.max(), 200)
    y_line = beta[0] + beta[1] * x_line

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(xhat, y, alpha=0.25, s=18, color="#2980B9", label="Player-seasons")
    ax.plot(x_line, y_line, color="#E74C3C", lw=2,
            label=f"OLS fit  (slope={beta[1]:.3f}, intercept={beta[0]:.3f})")
    ax.plot(x_line, x_line, color="grey", lw=1.2, linestyle="--", label="Perfect prediction (y=x)")
    ax.set_xlabel("xMB (Expected)", fontsize=11)
    ax.set_ylabel("MB (Actual)", fontsize=11)
    ax.set_title("MB vs xMB — Model 1 Fit", fontsize=12, fontweight="bold", color="#2C3E50")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf


def fig_residuals(clean):
    """Residual plot: (xMB - MB) vs MB."""
    y    = clean["MB"].values.astype(float)
    xhat = clean["xMB"].values.astype(float)
    resid = xhat - y

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(y, resid, alpha=0.25, s=15, color="#8E44AD")
    ax.axhline(0, color="black", lw=1.4, linestyle="--")
    ax.set_xlabel("MB (Actual)", fontsize=11)
    ax.set_ylabel("Residual  (xMB − MB)", fontsize=11)
    ax.set_title("Residual Plot — Model 1", fontsize=12, fontweight="bold", color="#2C3E50")
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf



In [101]:
# =============================================================================
# BUILD PDF
# =============================================================================

def build_pdf(clean, pe_stats, bayes_stats, output_path="mb_comparison_report.pdf"):

    doc = SimpleDocTemplate(
        output_path, pagesize=letter,
        leftMargin=0.75*inch, rightMargin=0.75*inch,
        topMargin=0.75*inch,  bottomMargin=0.75*inch
    )
    base = getSampleStyleSheet()
    S = {
        "title":   ParagraphStyle("T",    parent=base["Title"],
                                  fontSize=22, alignment=TA_CENTER, spaceAfter=6),
        "sub":     ParagraphStyle("Sub",  parent=base["Normal"],
                                  fontSize=11, alignment=TA_CENTER,
                                  textColor=colors.HexColor("#555555"), spaceAfter=4),
        "h1":      ParagraphStyle("H1",   parent=base["Heading1"],
                                  fontSize=14, textColor=colors.HexColor("#2C3E50"),
                                  spaceBefore=12, spaceAfter=6),
        "h2":      ParagraphStyle("H2",   parent=base["Heading2"],
                                  fontSize=11, textColor=colors.HexColor("#2980B9"),
                                  spaceBefore=8, spaceAfter=4),
        "body":    ParagraphStyle("Body", parent=base["Normal"],
                                  fontSize=9.5, leading=14, spaceAfter=5),
        "small":   ParagraphStyle("Sm",   parent=base["Normal"],
                                  fontSize=8.5, leading=12,
                                  textColor=colors.HexColor("#444444")),
        "caption": ParagraphStyle("Cap",  parent=base["Normal"],
                                  fontSize=8,   leading=11,
                                  textColor=colors.grey,
                                  alignment=TA_CENTER, spaceAfter=8),
        "header":  ParagraphStyle("TH",   parent=base["Normal"],
                                  textColor=colors.white,
                                  fontName="Helvetica-Bold",
                                  fontSize=9, alignment=TA_CENTER),
        "verdict": ParagraphStyle("V",    parent=base["Normal"],
                                  fontSize=11, leading=16,
                                  textColor=colors.HexColor("#1a5276"),
                                  fontName="Helvetica-Bold",
                                  alignment=TA_CENTER, spaceAfter=8),
    }

    story = []

    # ── Title ──────────────────────────────────────────────────────────────
    story.append(Spacer(1, 0.6*inch))
    story.append(Paragraph("MB vs xMB", S["title"]))
    story.append(Paragraph("Percent Error &amp; Bayesian Model Comparison Report", S["sub"]))
    story.append(Paragraph("Baseball Statcast  |  Seasons 2019–2024", S["sub"]))
    story.append(Spacer(1, 0.15*inch))
    story.append(divider())
    story.append(Spacer(1, 0.1*inch))
    story.append(Paragraph(
        "This report evaluates how well the <b>expected MB (xMB)</b> — derived from "
        "machine-learning predictions of individual counting stats — matches the "
        "<b>actual MB</b> computed from observed statistics. "
        "Two complementary tests are applied: a <b>Percent Error analysis</b> measuring "
        "prediction accuracy at the player level, and a <b>Bayesian Model Comparison</b> "
        "assessing whether xMB carries statistically meaningful predictive information "
        "about MB beyond a simple mean baseline.",
        S["body"]))
    story.append(PageBreak())

    # ── Section 1: Percent Error ───────────────────────────────────────────
    story.append(Paragraph("Section 1 — Percent Error Analysis", S["h1"]))
    story.append(divider())
    story.append(Paragraph(
        "Percent error measures the signed relative deviation of the expected value "
        "from the actual: <b>(xMB − MB) / |MB| × 100%</b>. "
        "A positive value means the model over-predicts; negative means it under-predicts.",
        S["body"]))
    story.append(Spacer(1, 0.1*inch))

    # Summary stats table
    sum_rows = [
        [th("Metric", S), th("Value", S)],
        ["Players analysed",           str(pe_stats["n"])],
        ["Mean Percent Error",          f"{pe_stats['mean_pe']}%  (+ = over-predict)"],
        ["Median Percent Error",        f"{pe_stats['median_pe']}%"],
        ["Mean Abs. Percent Error",     f"{pe_stats['mean_ape']}%"],
        ["Median Abs. Percent Error",   f"{pe_stats['median_ape']}%"],
        ["Std Dev of Percent Error",    f"{pe_stats['std_pe']}%"],
        ["Min Percent Error",           f"{pe_stats['min_pe']}%"],
        ["Max Percent Error",           f"{pe_stats['max_pe']}%"],
        ["Players within ±5%",          f"{pe_stats['pct_within_5']}%"],
        ["Players within ±10%",         f"{pe_stats['pct_within_10']}%"],
        ["Players within ±20%",         f"{pe_stats['pct_within_20']}%"],
    ]
    story.append(styled_table(sum_rows, [3.2*inch, 3.0*inch], header_color="#2C3E50"))
    story.append(Spacer(1, 0.2*inch))

    # Distribution figure
    story.append(Paragraph("Error Distribution &amp; Seasonal Breakdown", S["h2"]))
    buf_dist = fig_percent_error_dist(clean)
    story.append(rl_image(buf_dist, 6.5, fig_w=9, fig_h=4))
    story.append(Paragraph(
        "Left: Distribution of percent error across all player-seasons. "
        "Right: Box plots of percent error by season.", S["caption"]))
    story.append(Spacer(1, 0.15*inch))

    # Top over-predicted
    story.append(Paragraph("Top 5 Most Over-Predicted Players  (xMB &gt;&gt; MB)", S["h2"]))
    over = pe_stats["top_over"][["Name", "Season", "MB", "xMB", "Percent_Error"]].round(2)
    over_rows = [[th(c, S) for c in ["Name", "Season", "MB", "xMB", "% Error"]]]
    for _, row in over.iterrows():
        over_rows.append([str(row["Name"]), str(int(row["Season"])),
                          str(row["MB"]), str(row["xMB"]),
                          f"{row['Percent_Error']:.1f}%"])
    story.append(styled_table(over_rows,
        [2.2*inch, 0.8*inch, 0.9*inch, 0.9*inch, 1.0*inch],
        header_color="#C0392B"))
    story.append(Spacer(1, 0.15*inch))

    # Top under-predicted
    story.append(Paragraph("Top 5 Most Under-Predicted Players  (xMB &lt;&lt; MB)", S["h2"]))
    under = pe_stats["top_under"][["Name", "Season", "MB", "xMB", "Percent_Error"]].round(2)
    under_rows = [[th(c, S) for c in ["Name", "Season", "MB", "xMB", "% Error"]]]
    for _, row in under.iterrows():
        under_rows.append([str(row["Name"]), str(int(row["Season"])),
                           str(row["MB"]), str(row["xMB"]),
                           f"{row['Percent_Error']:.1f}%"])
    story.append(styled_table(under_rows,
        [2.2*inch, 0.8*inch, 0.9*inch, 0.9*inch, 1.0*inch],
        header_color="#1A5276"))
    story.append(PageBreak())

    # ── Section 2: Bayesian Model Comparison ──────────────────────────────
    story.append(Paragraph("Section 2 — Bayesian Model Comparison", S["h1"]))
    story.append(divider())
    story.append(Paragraph(
        "Two models of MB are compared using the <b>Bayesian Information Criterion (BIC)</b>. "
        "The BIC approximates each model's log marginal likelihood, penalising complexity. "
        "The <b>Bayes Factor (BF<sub>10</sub>)</b> is then derived as "
        "exp((BIC<sub>0</sub> − BIC<sub>1</sub>) / 2) and interpreted on Jeffreys' scale.",
        S["body"]))
    story.append(Spacer(1, 0.08*inch))

    # Model definitions
    model_def_rows = [
        [th("Model", S), th("Description", S), th("Parameters", S)],
        ["Model 0  (Null)", "MB ~ mean(MB)  — xMB has no predictive value", "1  (mean)"],
        ["Model 1  (xMB)",  "MB ~ a + b·xMB  — xMB linearly predicts MB",   "2  (intercept, slope)"],
    ]
    story.append(styled_table(model_def_rows,
        [1.5*inch, 3.8*inch, 1.4*inch], header_color="#2C3E50"))
    story.append(Spacer(1, 0.15*inch))

    # BIC & Bayes Factor results
    story.append(Paragraph("Results", S["h2"]))
    bf_rows = [
        [th("Statistic", S), th("Value", S)],
        ["BIC  — Model 0  (Null)",              str(bayes_stats["bic_0"])],
        ["BIC  — Model 1  (xMB predictor)",     str(bayes_stats["bic_1"])],
        ["Delta BIC  (BIC<sub>0</sub> − BIC<sub>1</sub>)", str(bayes_stats["delta_bic"])],
        ["Bayes Factor  (BF<sub>10</sub>)",     str(bayes_stats["bayes_factor"])],
        ["log<sub>10</sub>  Bayes Factor",      str(bayes_stats["log10_bf"])],
        ["Posterior P(Model 1 | data)",         f"{bayes_stats['posterior_m1']}%"],
        ["OLS Intercept  (a)",                  str(bayes_stats["intercept"])],
        ["OLS Slope  (b)",                      str(bayes_stats["slope"])],
        ["Model 1  R<super>2</super>",          str(bayes_stats["r2_m1"])],
    ]
    story.append(styled_table(bf_rows, [3.2*inch, 3.0*inch], header_color="#2980B9"))
    story.append(Spacer(1, 0.15*inch))

    # Jeffreys verdict
    story.append(Paragraph(
        f"Verdict (Jeffreys' Scale):  {bayes_stats['label']}", S["verdict"]))
    story.append(Spacer(1, 0.1*inch))

    # Jeffreys scale reference
    story.append(Paragraph("Jeffreys' Scale Reference", S["h2"]))
    jeff_rows = [
        [th("Bayes Factor (BF<sub>10</sub>)", S), th("Interpretation", S)],
        ["< 1",      "Against Model 1  (xMB uninformative)"],
        ["1 – 3",    "Anecdotal evidence for Model 1"],
        ["3 – 10",   "Moderate evidence for Model 1"],
        ["10 – 30",  "Strong evidence for Model 1"],
        ["30 – 100", "Very Strong evidence for Model 1"],
        ["> 100",    "Decisive evidence for Model 1"],
    ]
    # Highlight the row corresponding to the actual result
    bf_val = bayes_stats["bayes_factor"]
    highlight = None
    if bf_val < 1:      highlight = 1
    elif bf_val < 3:    highlight = 2
    elif bf_val < 10:   highlight = 3
    elif bf_val < 30:   highlight = 4
    elif bf_val < 100:  highlight = 5
    else:               highlight = 6

    story.append(styled_table(jeff_rows, [2.5*inch, 4.0*inch],
                              header_color="#2C3E50", highlight_row=highlight))
    story.append(Paragraph(
        "Green row = where your result falls on the scale.", S["caption"]))
    story.append(PageBreak())

    # ── Section 3: Visualisations ──────────────────────────────────────────
    story.append(Paragraph("Section 3 — Model Visualisations", S["h1"]))
    story.append(divider())

    story.append(Paragraph("MB vs xMB Scatter (Model 1 OLS Fit)", S["h2"]))
    buf_scatter = fig_mb_vs_xmb(clean)
    story.append(rl_image(buf_scatter, 6.0, fig_w=7, fig_h=5))
    story.append(Paragraph(
        "Each point is one player-season. The red line is the OLS fit from Model 1; "
        "the dashed grey line is perfect prediction (y = x). "
        "A slope near 1 and intercept near 0 indicates good calibration.",
        S["caption"]))
    story.append(Spacer(1, 0.2*inch))

    story.append(Paragraph("Residual Plot  (xMB − MB vs MB)", S["h2"]))
    buf_resid = fig_residuals(clean)
    story.append(rl_image(buf_resid, 6.0, fig_w=7, fig_h=4))
    story.append(Paragraph(
        "Residuals should be randomly scattered around zero with no obvious pattern. "
        "A funnel shape would indicate heteroskedasticity; a curved pattern would suggest "
        "a non-linear relationship that a linear model is missing.",
        S["caption"]))

    doc.build(story)
    print(f"\nPDF saved to: {output_path}")



In [102]:
# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    clean, pe_stats, bayes_stats = run_analysis("expected_all_stats.csv")
    build_pdf(clean, pe_stats, bayes_stats, "mb_comparison_report.pdf")
    clean[["Name", "Season", "MB", "xMB", "Percent_Error", "Abs_Percent_Error"]].to_csv(
        "mb_comparison_results.csv", index=False)
    print("Per-player results saved to: mb_comparison_results.csv")
    os.startfile('mb_comparison_report.pdf')

C:\Users\nncg7\AppData\Local\Temp\ipykernel_18556\1355482772.py:62: RuntimeWarning: overflow encountered in exp
  bayes_factor = np.exp(delta_bic / 2)
C:\Users\nncg7\AppData\Local\Temp\ipykernel_18556\1355482772.py:63: RuntimeWarning: invalid value encountered in scalar divide
  posterior_m1 = bayes_factor / (1 + bayes_factor)



PDF saved to: mb_comparison_report.pdf
Per-player results saved to: mb_comparison_results.csv
